# 🔮 FORESIGHT — 09: Model Explainability & SHAP Driver Decomposition

This notebook demonstrates:
- **TreeSHAP Global Feature Importance Rankings**
- **Feature Group Impact Breakdown** (Autoregressive vs Calendar vs Pricing)
- **Local Waterfall Attributions & Demand Decompositions**
- **Automated Business Narrative Generation**

In [ ]:
import pandas as pd
import numpy as np
from foresight.forecasting.base import BaseForecaster
from foresight.explainability.shap_explainer import ForecastExplainer

# 1. Load Data & Champion Model
df = pd.read_parquet('data/processed/features_engineered.parquet')
model = BaseForecaster.load('models/champion_forecaster.pkl')
print(f"Loaded champion model '{model.name}' with {len(model.feature_names_)} features.")

# 2. Initialize Explainer
explainer = ForecastExplainer(model)
sample_df = df.sample(1000, random_state=42).reset_index(drop=True)
importances = explainer.compute_global_importance(sample_df)

print("=== TOP 10 GLOBAL DEMAND DRIVERS ===")
for imp in importances[:10]:
    print(f"Rank {imp.rank:02d}: {imp.feature_name:<25} | Mean |SHAP| = {imp.mean_abs_shap:.3f} | {imp.relative_importance_pct:.1f}%")

## 3. Local Observation Attribution & Executive Narrative

In [ ]:
test_row = df[df['is_promoted'] == True].iloc[0]
exp = explainer.explain_observation(
    row_features=test_row,
    sku_id=str(test_row['sku_id']),
    store_id=str(test_row['store_id']),
    date=str(test_row['date']),
)

print(f"Baseline E[y]:    {exp.base_value:.1f} units")
print(f"Predicted y_hat:  {exp.predicted_value:.1f} units")
print(f"Executive Narrative:\n{exp.business_narrative}")